# UWF Zeek: fine-tuned LLM (Model 3)

Teaches Qwen2.5-7B-Instruct to read one 5-minute connection summary and give Sai's three answers:
`attack` (binary_target), `tactics` (tactic_targets) and `techniques` (mitre_targets).

- **Data:** `SPLIT` switch. Default `sai_files`: Sai's `train_edges.parquet` and `validation_edges.parquet` (her random
  80/10/10 window split, the same files as the Laya notebook). `test_edges.parquet` is scored only once it is available;
  without it the notebook trains and saves the model, and the scores are run later.
  The older `leave_one_dataset_out` / `year` / `random_window` options read `sai_edges_5min_capture.parquet` instead
- **Text:** plain-English summary of the connection plus what its source did in the same window; no IPs, times,
  dataset names or label columns
- **Before / after:** the untrained model is scored first, then fine-tuned with QLoRA and scored again on the same test set
- **Test sampling:** all attack combinations and normal traffic are sampled with weights, so detection scores reflect
  the test file's real attack rate

Run order: setup → settings → data → split → prompts → scoring → model → zero-shot → fine-tune → fine-tuned scores.

## 1. Setup

In [ ]:
import importlib.util, subprocess, sys

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    if importlib.util.find_spec("unsloth") is None:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "unsloth"])

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## 2. Settings

In [ ]:
import json, os, random, re, time
from collections import Counter
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)
try:
    display
except NameError:
    display = print

DRIVE_BASE = Path("/content/drive/Shareddrives/298B") if IN_COLAB else Path("./uwf_work")
CFG = {
    # split: "sai_files" (Sai's train / validation / test files, main), or one source file split here:
    # "leave_one_dataset_out", "year" (hold out a year group), "random_window" (Sai's random 80/10/10 windows)
    "SPLIT": "sai_files",
    # Sai's train_edges.parquet and validation_edges.parquet (raw units; a copy is also in laya/processed_edges)
    "SPLIT_DIR": str(DRIVE_BASE / "07 GraphSAGE - data prep artifacts" / "processed_edges"),
    "TEST_FILE": str(DRIVE_BASE / "07 GraphSAGE - data prep artifacts" / "processed_edges" / "test_edges.parquet"),                     # path of the test file once it is available; None = train only
    "SOURCE_FILE": str(DRIVE_BASE / "blocked_cv" / "sai_edges_5min_capture.parquet"),
    "OUT_DIR": str(DRIVE_BASE / "llm_finetune"),
    "HOLD_OUT": "UWF-ZeekDataSum25-1",       # leave_one_dataset_out / year only (or "2021-22" / "2024" / "2025")
    "PILOT": False,                         # True: small and fast end-to-end check
    "SEED": 42,
    # training set: cap per technique combination, the same number of normal connections, duplicates removed
    "TRAIN_CAP_PER_COMBO": 500,
    "NORMAL_PER_ATTACK": 1.0,
    "VAL_CAP_PER_COMBO": 50,                # sai_files: validation sampled the same way from the validation file
    "VAL_FRACTION": 0.1,                    # other splits: share of the training sample kept for validation
    # test set: up to this many per attack combination, plus this many normal connections (weighted back to real counts)
    "TEST_CAP_PER_COMBO": 1000,
    "TEST_NORMAL": 5000,
    # model and training
    "BASE_MODEL": "unsloth/Qwen2.5-7B-Instruct-bnb-4bit",
    "MAX_SEQ_LEN": 1024,
    "LORA_RANK": 16,
    "EPOCHS": 1,
    "LEARNING_RATE": 2e-4,
    "BATCH_SIZE": 8,
    "GRAD_ACCUM": 2,
    "INFER_BATCH": 32,
    "RUN_MODEL": IN_COLAB,                  # load the LLM and run zero-shot, fine-tuning and scoring
}
CFG.update(json.loads(os.environ.get("UWF_LLM_OVERRIDES", "{}")))
if CFG["PILOT"]:
    CFG.update({"TRAIN_CAP_PER_COMBO": 150, "VAL_CAP_PER_COMBO": 20, "TEST_CAP_PER_COMBO": 150, "TEST_NORMAL": 600,
                "EPOCHS": 1})

RUN_NAME = "sai_split" if CFG["SPLIT"] == "sai_files" else f"{CFG['SPLIT']}__{CFG['HOLD_OUT']}"
OUT_DIR = Path(CFG["OUT_DIR"]) / f"{RUN_NAME}{'__pilot' if CFG['PILOT'] else ''}"
OUT_DIR.mkdir(parents=True, exist_ok=True)
random.seed(CFG["SEED"]); np.random.seed(CFG["SEED"])
print("source:", CFG["SPLIT_DIR"] if CFG["SPLIT"] == "sai_files" else CFG["SOURCE_FILE"]); print("output:", OUT_DIR)

source: /content/drive/Shareddrives/298B/07 GraphSAGE - data prep artifacts/processed_edges
output: /content/drive/Shareddrives/298B/llm_finetune/sai_split


## 3. Connections as text

In [ ]:
import ast


def load_edges():
    if CFG["SPLIT"] != "sai_files":
        return pd.read_parquet(CFG["SOURCE_FILE"])
    files = {"train": Path(CFG["SPLIT_DIR"]) / "train_edges.parquet",
             "validation": Path(CFG["SPLIT_DIR"]) / "validation_edges.parquet", "test": Path(CFG["TEST_FILE"] or ".")}
    parts = []
    for split, file in files.items():
        if split == "test" and not CFG["TEST_FILE"] or not file.exists():
            assert split == "test", f"missing {file}: copy Sai's split files there or change SPLIT_DIR"
            print(f"no test file yet ({CFG['TEST_FILE'] or 'TEST_FILE not set'}): training only, scores are run once it is added")
            continue
        parts.append(pd.read_parquet(file).assign(split=split))
    return pd.concat(parts, ignore_index=True)


edges = load_edges()
edges["window_start"] = pd.to_datetime(edges["window_start"])
counts = edges["connection_count"].to_numpy(dtype=float)
assert (counts >= 1).all() and np.allclose(counts, np.round(counts)), \
    "connection counts are not whole numbers: these features look log-transformed or scaled, use the files saved before log1p"


def as_list(value):
    if isinstance(value, str):
        value = ast.literal_eval(value) if value.strip().startswith("[") else [value] if value.strip() else []
    if value is None or (isinstance(value, float) and np.isnan(value)):
        return []
    return sorted(str(v) for v in value)


edges["tactics"] = edges["tactic_targets"].map(as_list)
edges["techniques"] = edges["mitre_targets"].map(as_list)
edges["attack"] = edges["binary_target"].astype(int) == 1
edges["year_group"] = edges["window_start"].dt.year.map({2021: "2021-22", 2022: "2021-22", 2024: "2024", 2025: "2025"})

## what the source and destination did in the same window (the context the GNN gets from its graph)
window_key = [c for c in ["capture", "split"] if c in edges] + ["window_start"]
src_stats = edges.groupby(window_key + ["src_ip_zeek"]).agg(
    src_targets=("dest_ip_zeek", "nunique"), src_connections=("connection_count", "sum"),
    src_ports=("unique_destination_ports", "sum"))
dst_stats = edges.groupby(window_key + ["dest_ip_zeek"]).agg(dst_sources=("src_ip_zeek", "nunique"))
edges = edges.join(src_stats, on=window_key + ["src_ip_zeek"]).join(dst_stats, on=window_key + ["dest_ip_zeek"])

CONN_STATES = {
    "S0": "no reply to the connection attempt", "REJ": "connection rejected", "SF": "normal open and close",
    "RSTO": "reset by the originator", "RSTR": "reset by the responder", "RSTOS0": "originator sent SYN then reset",
    "RSTRH": "responder sent SYN-ACK then reset", "SH": "originator sent SYN then FIN", "SHR": "responder sent SYN-ACK then FIN",
    "S1": "established, not closed", "S2": "established, originator tried to close", "S3": "established, responder tried to close",
    "OTH": "no handshake seen",
}


def number(value):
    value = float(value or 0)
    return f"{value:,.0f}" if value >= 10 or value == int(value) else f"{value:.2f}"


def connection_text(row) -> str:
    service = row["most_common_service"] if isinstance(row["most_common_service"], str) and row["most_common_service"] else "unknown"
    state = row["most_common_connection_state"] if isinstance(row["most_common_connection_state"], str) else "unknown"
    port = row["most_common_destination_port"]
    port = "unknown" if port is None or (isinstance(port, float) and np.isnan(port)) else f"{int(port)}"
    return "\n".join([
        "Connection summary for one source-destination pair over 5 minutes:",
        f"- connections: {number(row['connection_count'])} (TCP {number(row['tcp_count'])}, UDP {number(row['udp_count'])}, ICMP {number(row['icmp_count'])})",
        f"- bytes sent: {number(row['total_orig_bytes'])}, bytes received: {number(row['total_resp_bytes'])}",
        f"- packets sent: {number(row['total_orig_packets'])}, packets received: {number(row['total_resp_packets'])}",
        f"- IP bytes sent: {number(row['total_orig_ip_bytes'])}, received: {number(row['total_resp_ip_bytes'])}, missed bytes: {number(row['total_missed_bytes'])}",
        f"- duration: total {number(row['total_duration'])} s, average {number(row['average_duration'])} s, longest {number(row['maximum_duration'])} s",
        f"- most common destination port: {port}; distinct destination ports: {number(row['unique_destination_ports'])}",
        f"- most common service: {service}",
        f"- most common connection state: {state} ({CONN_STATES.get(state, 'unknown state')})",
        f"- same 5 minutes, source overall: {number(row['src_targets'])} hosts contacted, {number(row['src_connections'])} connections",
        f"- same 5 minutes, destination overall: contacted by {number(row['dst_sources'])} sources",
    ])


def answer_json(row) -> str:
    return json.dumps({"attack": bool(row["attack"]), "tactics": row["tactics"], "techniques": row["techniques"]})


edges["text"] = [connection_text(r) for r in edges.to_dict("records")]
edges["answer"] = [answer_json(r) for r in edges.to_dict("records")]
edges["combo"] = np.where(edges["attack"],
                          [("T:" + ",".join(t)) if t else ("tactic-only:" + ",".join(a)) for t, a in zip(edges.techniques, edges.tactics)],
                          "normal")
print(f"{len(edges):,} connections | attacks: {int(edges.attack.sum()):,}",
      edges.groupby("split").attack.agg(["size", "sum"]).to_dict("index") if "split" in edges else f"| datasets: {edges.capture.nunique()}")
print("\nExample attack:\n" + edges.loc[edges.attack].iloc[0]["text"] + "\n→ " + edges.loc[edges.attack].iloc[0]["answer"])
print("\nExample normal:\n" + edges.loc[~edges.attack].iloc[0]["text"] + "\n→ " + edges.loc[~edges.attack].iloc[0]["answer"])

678,054 connections | attacks: 54,650 {'test': {'size': 69161, 'sum': 5670}, 'train': {'size': 541763, 'sum': 43367}, 'validation': {'size': 67130, 'sum': 5613}}

Example attack:
Connection summary for one source-destination pair over 5 minutes:
- connections: 1 (TCP 1, UDP 0, ICMP 0)
- bytes sent: 0, bytes received: 0
- packets sent: 6, packets received: 3
- IP bytes sent: 252, received: 132, missed bytes: 0
- duration: total 0.00 s, average 0.00 s, longest 0.00 s
- most common destination port: 53; distinct destination ports: 1
- most common service: unknown
- most common connection state: RSTO (reset by the originator)
- same 5 minutes, source overall: 4 hosts contacted, 5 connections
- same 5 minutes, destination overall: contacted by 1 sources
→ {"attack": true, "tactics": ["Reconnaissance"], "techniques": ["T1595"]}

Example normal:
Connection summary for one source-destination pair over 5 minutes:
- connections: 10 (TCP 0, UDP 10, ICMP 0)
- bytes sent: 1,760, bytes received: 1,7

## 4. Split, training set and weighted test set

In [ ]:
from sklearn.model_selection import train_test_split


def split_masks(df):
    """Returns (train_pool, test) boolean masks for the chosen SPLIT."""
    if CFG["SPLIT"] == "sai_files":
        return (df.split == "train").to_numpy(), (df.split == "test").to_numpy()
    if CFG["SPLIT"] == "leave_one_dataset_out":
        test = (df.capture == CFG["HOLD_OUT"]).to_numpy()
    elif CFG["SPLIT"] == "year":
        test = (df.year_group == CFG["HOLD_OUT"]).to_numpy()
    elif CFG["SPLIT"] == "random_window":
        ## Sai's split: whole 5-minute windows shuffled 80/10/10, stratified by whether the window has an attack
        windows = df.groupby("window_start").attack.max().reset_index()
        train_w, rest_w = train_test_split(windows, test_size=0.2, random_state=CFG["SEED"], stratify=windows.attack)
        _, test_w = train_test_split(rest_w, test_size=0.5, random_state=CFG["SEED"], stratify=rest_w.attack)
        test = df.window_start.isin(set(test_w.window_start)).to_numpy()
        return df.window_start.isin(set(train_w.window_start)).to_numpy(), test
    else:
        raise ValueError(CFG["SPLIT"])
    return ~test, test


train_pool_mask, test_mask = split_masks(edges)
HAS_TEST = bool(test_mask.any())
assert HAS_TEST or CFG["SPLIT"] == "sai_files", f"nothing to test: check HOLD_OUT={CFG['HOLD_OUT']!r}"
pool, test_pool = edges[train_pool_mask], edges[test_mask]


def capped_sample(frame, cap):
    """Up to `cap` connections per attack combination, the same number of normal ones, duplicate texts removed
    (keeping the majority answer), shuffled."""
    parts = [g.sample(min(len(g), cap), random_state=CFG["SEED"]) for _, g in frame[frame.attack].groupby("combo")]
    attacks = pd.concat(parts) if parts else frame.iloc[0:0]
    normal = frame[~frame.attack]
    normals = normal.sample(min(len(normal), int(len(attacks) * CFG["NORMAL_PER_ATTACK"]) or 1), random_state=CFG["SEED"])
    sample = pd.concat([attacks, normals])
    majority = sample.groupby("text").answer.agg(lambda s: s.value_counts().index[0])
    sample = sample.drop_duplicates("text").assign(answer=lambda d: d.text.map(majority))
    return sample.sample(frac=1.0, random_state=CFG["SEED"])


if CFG["SPLIT"] == "sai_files":
    ## training from Sai's training file, validation from her validation file (texts already in training dropped)
    train_set = capped_sample(pool, CFG["TRAIN_CAP_PER_COMBO"])
    val_set = capped_sample(edges[edges.split == "validation"], CFG["VAL_CAP_PER_COMBO"])
    val_set = val_set[~val_set.text.isin(set(train_set.text))]
else:
    train_all = capped_sample(pool, CFG["TRAIN_CAP_PER_COMBO"])
    n_val = max(1, int(len(train_all) * CFG["VAL_FRACTION"]))
    val_set, train_set = train_all.iloc[:n_val], train_all.iloc[n_val:]

## test set: sample each stratum (attack combination or normal) and keep a weight = stratum size / sample size
test_parts = []
for combo, group in test_pool.groupby("combo"):
    cap = CFG["TEST_NORMAL"] if combo == "normal" else CFG["TEST_CAP_PER_COMBO"]
    sample = group.sample(min(len(group), cap), random_state=CFG["SEED"])
    test_parts.append(sample.assign(weight=len(group) / len(sample)))
test_set = pd.concat(test_parts) if test_parts else test_pool.assign(weight=1.0)

## techniques the model can learn: present in the training pool
learnable = set(t for ts in pool.loc[pool.attack, "techniques"] for t in ts)
summary = pd.DataFrame({
    "connections": [len(train_set), len(val_set), len(test_set)],
    "attacks": [int(train_set.attack.sum()), int(val_set.attack.sum()), int(test_set.attack.sum())],
    "represents (weighted)": [len(train_set), len(val_set), int(test_set.weight.sum())],
}, index=["train", "validation", "test"])
display(summary)
print("attack combinations in training:", train_set[train_set.attack].combo.nunique(),
      "| techniques in held-out data:", sorted(set(t for ts in test_pool.techniques for t in ts)),
      "| of which learnable:", sorted(set(t for ts in test_pool.techniques for t in ts) & learnable))
for name, frame in [("train", train_set), ("val", val_set)] + [("test", test_set)] * HAS_TEST:
    frame.drop(columns=["tactic_targets", "mitre_targets"]).to_parquet(OUT_DIR / f"{name}.parquet", index=False)

,connections,attacks,represents (weighted)
train,4998,2295,4998
validation,475,249,475
test,7882,2882,69160


attack combinations in training: 49 | techniques in held-out data: ['T1016', 'T1018', 'T1046', 'T1048', 'T1059', 'T1078', 'T1110', 'T1190', 'T1203', 'T1210', 'T1548', 'T1587', 'T1589', 'T1590', 'T1592', 'T1595'] | of which learnable: ['T1016', 'T1018', 'T1046', 'T1048', 'T1059', 'T1078', 'T1110', 'T1190', 'T1203', 'T1210', 'T1548', 'T1587', 'T1589', 'T1590', 'T1592', 'T1595']


In [ ]:
## compare the synthetic test file with Sai's training file
feats = ["connection_count", "total_orig_bytes", "total_resp_bytes", "total_duration",
         "unique_destination_ports", "src_targets", "src_connections", "dst_sources"]
display(edges[edges.split.isin(["train", "test"])].groupby(["split", "attack"])[feats].median().T)

for split in ["train", "test"]:
    e = edges[edges.split == split]
    w = e.groupby("window_start").attack.agg(["min", "max"])
    mixed = w[(w["min"] == 0) & (w["max"] == 1)].index
    print(f"{split}: rows {len(e):,} | windows {e.window_start.nunique():,} | attack rate {e.attack.mean():.3f}"
          f" | attack rows in windows that also have normal traffic {e[e.attack & e.window_start.isin(mixed)].shape[0] / max(e.attack.sum(), 1):.3f}")
    print("   top services:", e.most_common_service.value_counts().head(5).to_dict())
print("test texts identical to a training text:", round(test_set.text.isin(set(train_set.text)).mean(), 3))

split                           test                 train          
attack                         False     True        False     True 
connection_count            3.000000  2.000000    3.000000  2.000000
total_orig_bytes          215.000000  0.000000  215.000000  0.000000
total_resp_bytes            0.000000  0.000000    0.000000  0.000000
total_duration              0.000298  0.002415    0.000354  0.002411
unique_destination_ports    1.000000  1.000000    1.000000  1.000000
src_targets                 1.000000  4.000000    1.000000  4.000000
src_connections             8.000000  9.000000    8.000000  8.000000
dst_sources                 2.000000  1.000000    2.000000  1.000000

train: rows 541,763 | windows 20,149 | attack rate 0.080 | attack rows in windows that also have normal traffic 0.007
   top services: {'dns': 434004, 'unknown': 84480, 'ntp': 6884, 'ssl': 5791, 'dhcp': 5715}
test: rows 69,161 | windows 2,518 | attack rate 0.082 | attack rows in windows that also have normal traffic 0.001
   top services: {'dns': 55564, 'unknown': 10792, 'ntp': 767, 'ssl': 758, 'dhcp': 668}
test texts identical to a training text: 0.539


## 5. Prompt and answer format

In [ ]:
label_rows = edges[edges.split != "test"] if "split" in edges else edges
TACTICS = sorted(set(t for ts in label_rows.tactics for t in ts))
TECHNIQUES = sorted(set(t for ts in label_rows.techniques for t in ts))
TECHNIQUE_NAMES = {
    "T1016": "System Network Configuration Discovery", "T1018": "Remote System Discovery", "T1046": "Network Service Discovery",
    "T1048": "Exfiltration Over Alternative Protocol", "T1059": "Command and Scripting Interpreter", "T1071": "Application Layer Protocol",
    "T1078": "Valid Accounts", "T1110": "Brute Force", "T1112": "Modify Registry", "T1133": "External Remote Services",
    "T1136": "Create Account", "T1190": "Exploit Public-Facing Application", "T1203": "Exploitation for Client Execution",
    "T1204": "User Execution", "T1210": "Exploitation of Remote Services", "T1505": "Server Software Component",
    "T1546": "Event Triggered Execution", "T1547": "Boot or Logon Autostart Execution", "T1548": "Abuse Elevation Control Mechanism",
    "T1557": "Adversary-in-the-Middle", "T1566": "Phishing", "T1571": "Non-Standard Port", "T1587": "Develop Capabilities",
    "T1589": "Gather Victim Identity Information", "T1590": "Gather Victim Network Information",
    "T1592": "Gather Victim Host Information", "T1595": "Active Scanning",
}
SYSTEM_PROMPT = (
    "You are a network security analyst. You read a summary of the Zeek connection logs between one source and one "
    "destination over 5 minutes and decide whether it is part of an attack.\n"
    "Answer with JSON only, in this exact form: "
    '{"attack": true or false, "tactics": [...], "techniques": [...]}.\n'
    "For normal traffic both lists are empty. An attack can have more than one tactic and technique; list technique "
    "IDs only, and leave techniques empty if none applies.\n"
    f"Allowed tactics: {', '.join(TACTICS)}.\n"
    "Allowed techniques: " + "; ".join(f"{t} {TECHNIQUE_NAMES.get(t, '')}".strip() for t in TECHNIQUES) + "."
)
ANSWER_PREFIX = '{"attack":'


def messages(text, answer=None):
    turns = [{"role": "system", "content": SYSTEM_PROMPT}, {"role": "user", "content": text}]
    if answer is not None:
        turns.append({"role": "assistant", "content": answer})
    return turns


def parse_answer(raw: str) -> dict:
    """Reads the model's JSON; anything outside the allowed labels is kept separately as invalid."""
    match = re.search(r"\{.*\}", raw or "", flags=re.S)
    result = {"attack": None, "tactics": [], "techniques": [], "invalid": [], "parsed": False}
    if not match:
        return result
    try:
        payload = json.loads(match.group(0))
    except json.JSONDecodeError:
        return result
    result["parsed"] = True
    result["attack"] = bool(payload.get("attack")) if isinstance(payload.get("attack"), bool) else None
    for key, allowed in [("tactics", TACTICS), ("techniques", TECHNIQUES)]:
        values = payload.get(key) or []
        values = values if isinstance(values, list) else [values]
        result[key] = sorted({str(v) for v in values if str(v) in allowed})
        result["invalid"] += [str(v) for v in values if str(v) not in allowed]
    return result


print(SYSTEM_PROMPT[:600] + " ...")
print(f"\nprompt length ≈ {len(SYSTEM_PROMPT) // 4 + 250} tokens")

You are a network security analyst. You read a summary of the Zeek connection logs between one source and one destination over 5 minutes and decide whether it is part of an attack.
Answer with JSON only, in this exact form: {"attack": true or false, "tactics": [...], "techniques": [...]}.
For normal traffic both lists are empty. An attack can have more than one tactic and technique; list technique IDs only, and leave techniques empty if none applies.
Allowed tactics: Collection, Command and Control, Credential Access, Defense Evasion, Discovery, Execution, Exfiltration, Initial Access, Lateral ...

prompt length ≈ 646 tokens


## 6. Scoring

In [ ]:
from sklearn.metrics import average_precision_score, f1_score, roc_auc_score
import numpy as np
import pandas as pd

def score_predictions(pred: pd.DataFrame, frame: pd.DataFrame) -> dict:
    """pred: index aligned with frame; columns p_attack, attack_pred, tactics_pred, techniques_pred, parsed, invalid."""
    w = frame["weight"].to_numpy() if "weight" in frame else np.ones(len(frame))
    y = frame["attack"].to_numpy().astype(int)
    p = pred["p_attack"].to_numpy()
    yhat = pred["attack_pred"].fillna(False).to_numpy().astype(bool)
    tp, fp = float(w[yhat & (y == 1)].sum()), float(w[yhat & (y == 0)].sum())
    fn, tn = float(w[~yhat & (y == 1)].sum()), float(w[~yhat & (y == 0)].sum())
    scores = {
        "parsed_answers": float(pred["parsed"].mean()),
        "answers_with_invalid_labels": float((pred["invalid"].map(len) > 0).mean()),
        "attack_pr_auc": float(average_precision_score(y, p, sample_weight=w)) if 0 < y.sum() < len(y) else np.nan,
        "attack_roc_auc": float(roc_auc_score(y, p, sample_weight=w)) if 0 < y.sum() < len(y) else np.nan,
        "attack_recall": tp / max(tp + fn, 1e-9), "attack_precision": tp / max(tp + fp, 1e-9),
        "false_alarm_rate": fp / max(fp + tn, 1e-9),
    }
    attacks = frame["attack"].to_numpy()
    for key, classes in [("tactics", TACTICS), ("techniques", TECHNIQUES)]:
        true_lists = [list(x) if isinstance(x, (list, np.ndarray, set)) else [] for x in frame.loc[attacks, key]]
        pred_lists = [list(x) if isinstance(x, (list, np.ndarray, set)) else [] for x in pred.loc[attacks, f"{key}_pred"]]
        keep = [i for i, t in enumerate(true_lists) if t]
        if not keep:
            continue
        index = {c: i for i, c in enumerate(classes)}
        Y = np.zeros((len(keep), len(classes))); P = np.zeros_like(Y)
        for row, i in enumerate(keep):
            for c in true_lists[i]: Y[row, index[c]] = 1
            for c in pred_lists[i]: P[row, index[c]] = 1
        present = Y.sum(0) > 0
        scores[f"{key}_exact_match"] = float(np.mean([set(true_lists[i]) == set(pred_lists[i]) for i in keep]))
        scores[f"{key}_first_correct"] = float(np.mean([len(pred_lists[i]) > 0 and pred_lists[i][0] in true_lists[i] for i in keep]))
        scores[f"{key}_micro_f1"] = float(f1_score(Y, P, average="micro", zero_division=0))
        scores[f"{key}_macro_f1"] = float(f1_score(Y[:, present], P[:, present], average="macro", zero_division=0))
    return scores

## 7. Load the base model (Qwen2.5-7B-Instruct, 4-bit)

In [ ]:
if CFG["RUN_MODEL"]:
    import torch
    from unsloth import FastLanguageModel

    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name=CFG["BASE_MODEL"], max_seq_length=CFG["MAX_SEQ_LEN"], load_in_4bit=True, dtype=None)
    tokenizer.padding_side = "left"
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    TRUE_ID = tokenizer.encode(" true", add_special_tokens=False)[0]
    FALSE_ID = tokenizer.encode(" false", add_special_tokens=False)[0]
    print("loaded", CFG["BASE_MODEL"], "| true/false token ids:", TRUE_ID, FALSE_ID)


def run_model(frame: pd.DataFrame, label: str) -> pd.DataFrame:
    """Greedy answers for every row; p_attack = P(' true') vs P(' false') right after '{"attack":'."""
    out_file = OUT_DIR / f"predictions_{label}.parquet"
    if out_file.exists():
        return pd.read_parquet(out_file).set_index("row")
    FastLanguageModel.for_inference(model)
    texts = frame["text"].tolist()
    unique = list(dict.fromkeys(texts))                     # score each distinct text once
    results = {}
    started = time.time()
    for start in range(0, len(unique), CFG["INFER_BATCH"]):
        batch = unique[start:start + CFG["INFER_BATCH"]]
        prompts = [tokenizer.apply_chat_template(messages(t), tokenize=False, add_generation_prompt=True) + ANSWER_PREFIX
                   for t in batch]
        inputs = tokenizer(prompts, return_tensors="pt", padding=True).to(model.device)
        with torch.no_grad():
            generated = model.generate(**inputs, max_new_tokens=96, do_sample=False, output_scores=True,
                                       return_dict_in_generate=True, pad_token_id=tokenizer.pad_token_id)
        first = torch.softmax(generated.scores[0][:, [TRUE_ID, FALSE_ID]].float(), dim=-1)[:, 0].cpu().numpy()
        new_tokens = generated.sequences[:, inputs["input_ids"].shape[1]:]
        for text, prob, tokens in zip(batch, first, new_tokens):
            raw = ANSWER_PREFIX + tokenizer.decode(tokens, skip_special_tokens=True)
            results[text] = (float(prob), raw)
        if start // CFG["INFER_BATCH"] % 20 == 0:
            print(f"  {label}: {start + len(batch):,}/{len(unique):,} distinct texts ({(time.time() - started) / 60:.1f} min)")
    rows = []
    for row, text in zip(frame.index, texts):
        prob, raw = results[text]
        parsed = parse_answer(raw)
        rows.append({"row": row, "p_attack": prob,
                     "attack_pred": parsed["attack"] if parsed["attack"] is not None else prob >= 0.5,
                     "tactics_pred": parsed["tactics"], "techniques_pred": parsed["techniques"],
                     "parsed": parsed["parsed"], "invalid": parsed["invalid"], "raw": raw})
    pred = pd.DataFrame(rows).set_index("row")
    pred.reset_index().to_parquet(out_file, index=False)
    return pred

==((====))==  Unsloth 2026.10.11: Fast Qwen2 patching. Transformers: 5.17.0.
   \\   /|    NVIDIA A100-SXM4-80GB. Num GPUs = 1. Max memory: 79.251 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu130. CUDA: 8.0. CUDA Toolkit: 13.0. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

loaded unsloth/Qwen2.5-7B-Instruct-bnb-4bit | true/false token ids: 830 895


## 8. Before fine-tuning: zero-shot scores

In [ ]:
if CFG["RUN_MODEL"] and HAS_TEST:
    assert not any("lora_" in n for n, _ in model.named_parameters()), \
        "the model already has the fine-tuned add-on: restart the runtime and run sections 1-7 before this cell"

    # Remove stale cached predictions if they do not match the expected length of the test_set
    zero_shot_cache = OUT_DIR / "predictions_zero_shot.parquet"
    if zero_shot_cache.exists():
        cached_len = len(pd.read_parquet(zero_shot_cache))
        if cached_len != len(test_set):
            print(f"Removing stale cache file with {cached_len} rows (expected {len(test_set)} rows)")
            zero_shot_cache.unlink()

    zero_pred = run_model(test_set, "zero_shot")
    zero_scores = score_predictions(zero_pred, test_set)
    json.dump(zero_scores, open(OUT_DIR / "scores_zero_shot.json", "w"), indent=1)
    display(pd.Series(zero_scores).round(4))
elif CFG["RUN_MODEL"]:
    print("no test file yet: zero-shot scores skipped")

,0
parsed_answers,1.0000
answers_with_invalid_labels,0.0089
attack_pr_auc,0.0584
attack_roc_auc,0.3198
attack_recall,0.0002
attack_precision,0.0004
false_alarm_rate,0.0410
tactics_exact_match,0.0000
tactics_first_correct,0.0000
tactics_micro_f1,0.0007


## 9. Fine-tune (QLoRA: compressed base model + small trainable add-on)

In [ ]:
ADAPTER_DIR = OUT_DIR / "adapter"

if CFG["RUN_MODEL"] and not (ADAPTER_DIR / "adapter_config.json").exists():
    from datasets import Dataset
    from trl import SFTConfig, SFTTrainer
    from unsloth.chat_templates import train_on_responses_only

    model = FastLanguageModel.get_peft_model(
        model, r=CFG["LORA_RANK"], lora_alpha=CFG["LORA_RANK"], lora_dropout=0, bias="none",
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
        use_gradient_checkpointing="unsloth", random_state=CFG["SEED"])
    tokenizer.padding_side = "right"

    def to_dataset(frame):
        texts = [tokenizer.apply_chat_template(messages(t, a), tokenize=False) for t, a in zip(frame.text, frame.answer)]
        return Dataset.from_dict({"text": texts})

    trainer = SFTTrainer(
        model=model, tokenizer=tokenizer,
        train_dataset=to_dataset(train_set), eval_dataset=to_dataset(val_set),
        args=SFTConfig(
            dataset_text_field="text", max_seq_length=CFG["MAX_SEQ_LEN"],
            per_device_train_batch_size=CFG["BATCH_SIZE"], gradient_accumulation_steps=CFG["GRAD_ACCUM"],
            num_train_epochs=CFG["EPOCHS"], learning_rate=CFG["LEARNING_RATE"], warmup_ratio=0.03,
            lr_scheduler_type="linear", optim="adamw_8bit", weight_decay=0.01,
            logging_steps=20, eval_strategy="steps", eval_steps=200, save_strategy="no",
            output_dir=str(OUT_DIR / "trainer"), seed=CFG["SEED"], report_to="none"),
    )
    ## learn only from the answer, not from the prompt
    trainer = train_on_responses_only(trainer, instruction_part="<|im_start|>user\n",
                                      response_part="<|im_start|>assistant\n")
    started = time.time()
    trainer.train()
    print(f"training took {(time.time() - started) / 60:.1f} min")
    pd.DataFrame(trainer.state.log_history).to_csv(OUT_DIR / "training_log.csv", index=False)
    model.save_pretrained(str(ADAPTER_DIR)); tokenizer.save_pretrained(str(ADAPTER_DIR))
    print("saved adapter to", ADAPTER_DIR)
    tokenizer.padding_side = "left"
elif CFG["RUN_MODEL"]:
    ## already trained in an earlier session: load base model + saved add-on
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name=str(ADAPTER_DIR), max_seq_length=CFG["MAX_SEQ_LEN"], load_in_4bit=True, dtype=None)
    tokenizer.padding_side = "left"
    print("loaded saved adapter from", ADAPTER_DIR)

==((====))==  Unsloth 2026.10.11: Fast Qwen2 patching. Transformers: 5.17.0.
   \\   /|    NVIDIA A100-SXM4-80GB. Num GPUs = 1. Max memory: 79.251 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu130. CUDA: 8.0. CUDA Toolkit: 13.0. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

Unsloth: compiling each transformer block as one region (Qwen2DecoderLayer).
loaded saved adapter from /content/drive/Shareddrives/298B/llm_finetune/sai_split/adapter


## 10. After fine-tuning: scores and comparison

In [ ]:
def per_technique(pred: pd.DataFrame, frame: pd.DataFrame) -> pd.DataFrame:
    rows = []
    attacks = frame["attack"].to_numpy()
    for technique in sorted(set(t for ts in frame.loc[attacks, "techniques"] for t in ts)):
        mask = frame["techniques"].map(lambda ts: technique in ts).to_numpy() & attacks
        found = pred.loc[mask, "techniques_pred"].map(lambda ts: technique in ts)
        rows.append({"technique": technique, "test_connections": int(mask.sum()),
                     "learnable (in training data)": technique in learnable, "recall": float(found.mean())})
    return pd.DataFrame(rows).sort_values("test_connections", ascending=False)

if CFG["RUN_MODEL"] and HAS_TEST:
    # Remove stale cached fine-tuned predictions if they do not match the test set length
    tuned_cache = OUT_DIR / "predictions_fine_tuned.parquet"
    if tuned_cache.exists():
        cached_len = len(pd.read_parquet(tuned_cache))
        if cached_len != len(test_set):
            print(f"Removing stale fine-tuned cache file with {cached_len} rows (expected {len(test_set)} rows)")
            tuned_cache.unlink()

    tuned_pred = run_model(test_set, "fine_tuned")
    tuned_scores = score_predictions(tuned_pred, test_set)
    json.dump(tuned_scores, open(OUT_DIR / "scores_fine_tuned.json", "w"), indent=1)

    comparison = pd.DataFrame({"zero-shot": zero_scores, "fine-tuned": tuned_scores})
    print(f"=== {CFG['SPLIT']} | held out: {CFG['HOLD_OUT']} | pilot: {CFG['PILOT']} ===")
    display(comparison.round(4))
    print("\nper technique (fine-tuned):")
    display(per_technique(tuned_pred, test_set).round(4))
    comparison.to_csv(OUT_DIR / "comparison.csv")
    print("\nexample answers:")
    for i in test_set.index[:3]:
        print("  true:", test_set.loc[i, "answer"], "\n  pred:", tuned_pred.loc[i, "raw"][:160])
elif CFG["RUN_MODEL"]:
    print("fine-tuning done; no test file yet. Add it at TEST_FILE, restart the runtime and run all cells:")
    print("training is skipped (saved adapter), zero-shot and fine-tuned scores are run.")

Both `max_new_tokens` (=96) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Removing stale fine-tuned cache file with 894 rows (expected 7882 rows)


Both `max_new_tokens` (=96) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


  fine_tuned: 32/4,127 distinct texts (0.1 min)


Both `max_new_tokens` (=96) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=96) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=96) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=96) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generati

  fine_tuned: 672/4,127 distinct texts (1.4 min)


... (12 outputs left out)

,zero-shot,fine-tuned
parsed_answers,1.0000,1.0000
answers_with_invalid_labels,0.0089,0.0000
attack_pr_auc,0.0584,0.9868
attack_roc_auc,0.3198,0.9993
attack_recall,0.0002,0.9984
attack_precision,0.0004,0.8289
false_alarm_rate,0.0410,0.0184
tactics_exact_match,0.0000,0.9334
tactics_first_correct,0.0000,0.9483
tactics_micro_f1,0.0007,0.9539



per technique (fine-tuned):


,technique,test_connections,learnable (in training data),recall
15,T1595,1078,True,0.9369
5,T1078,594,True,0.9933
6,T1110,555,True,1.0000
7,T1190,462,True,0.9978
3,T1048,47,True,0.6809
13,T1590,24,True,0.7083
14,T1592,24,True,0.7083
12,T1589,8,True,0.0000
11,T1587,7,True,0.1429
10,T1548,4,True,0.0000



example answers:
  true: {"attack": true, "tactics": ["Discovery"], "techniques": ["T1016"]} 
  pred: {"attack": true, "tactics": ["Discovery"], "techniques": ["T1046"]}
  true: {"attack": true, "tactics": ["Discovery"], "techniques": ["T1018"]} 
  pred: {"attack": true, "tactics": ["Initial Access", "Reconnaissance"], "techniques": ["T1190", "T1595"]}
  true: {"attack": true, "tactics": ["Discovery"], "techniques": ["T1018"]} 
  pred: {"attack": true, "tactics": ["Defense Evasion", "Reconnaissance"], "techniques": ["T1078", "T1595"]}


In [ ]:
## check: did training change the model, and which model made each prediction file?
log = pd.read_csv(OUT_DIR / "training_log.csv")
print(log[[c for c in ["step", "loss", "eval_loss"] if c in log]].dropna(how="all").to_string())
if (OUT_DIR / "predictions_fine_tuned.parquet").exists() and (OUT_DIR / "predictions_zero_shot.parquet").exists():
    z = pd.read_parquet(OUT_DIR / "predictions_zero_shot.parquet")
    f = pd.read_parquet(OUT_DIR / "predictions_fine_tuned.parquet")
    print("\nshare of identical raw answers:", (z.raw.values == f.raw.values).mean())
    print("identical attack probabilities:", np.allclose(z.p_attack, f.p_attack))

for name in ["predictions_zero_shot.parquet", "adapter/adapter_config.json", "predictions_fine_tuned.parquet"]:
    path = OUT_DIR / name
    print(f"{name:35s}", time.ctime(path.stat().st_mtime) if path.exists() else "missing")

lora_b = [p.detach().abs().sum().item() for n, p in model.named_parameters() if "lora_B" in n]
print("\nLoRA add-on layers that are non-zero:", sum(v > 0 for v in lora_b), "of", len(lora_b))

    step      loss  eval_loss
0     20  0.221574        NaN
1     40  0.081876        NaN
2     60  0.045827        NaN
3     80  0.046350        NaN
4    100  0.037037        NaN
5    120  0.033010        NaN
6    140  0.021981        NaN
7    160  0.022619        NaN
8    180  0.021099        NaN
9    200  0.019887        NaN
10   200       NaN   0.021761
11   220  0.011888        NaN
12   240  0.017923        NaN
13   260  0.018927        NaN
14   280  0.014087        NaN
15   300  0.015390        NaN
16   313       NaN   0.016919
17   313       NaN        NaN

share of identical raw answers: 0.5978178127378838
identical attack probabilities: False
predictions_zero_shot.parquet       Sat Oct 31 21:25:32 2026
adapter/adapter_config.json         Sat Oct 31 20:42:26 2026
predictions_fine_tuned.parquet      Sat Oct 31 22:27:55 2026

LoRA add-on layers that are non-zero: 196 of 196


In [ ]:
## extra metrics: leakage, F1, alerts per attack, 95% ranges
from sklearn.metrics import average_precision_score

print("test texts identical to a training text:", round(test_set.text.isin(set(train_set.text)).mean(), 3))

def detection(pred, frame):
    w, y = frame.weight.to_numpy(), frame.attack.to_numpy().astype(int)
    yhat = pred.loc[frame.index, "attack_pred"].fillna(False).to_numpy().astype(bool)
    tp, fp, fn = w[yhat & (y == 1)].sum(), w[yhat & (y == 0)].sum(), w[~yhat & (y == 1)].sum()
    tn = w[~yhat & (y == 0)].sum()
    rec, prec = tp / max(tp + fn, 1e-9), tp / max(tp + fp, 1e-9)
    return {"pr_auc": average_precision_score(y, pred.loc[frame.index, "p_attack"], sample_weight=w),
            "recall": rec, "precision": prec, "f1": 2 * prec * rec / max(prec + rec, 1e-9),
            "false_alarm_rate": fp / max(fp + tn, 1e-9), "alerts_per_real_attack": (tp + fp) / max(tp, 1e-9)}

rng = np.random.default_rng(CFG["SEED"])
for label, pred in [("zero-shot", zero_pred), ("fine-tuned", tuned_pred)]:
    point = detection(pred, test_set)
    boots = pd.DataFrame([detection(pred, test_set.iloc[rng.integers(0, len(test_set), len(test_set))]) for _ in range(200)])
    print(f"\n{label}")
    display(pd.DataFrame({"value": point, "95% low": boots.quantile(0.025), "95% high": boots.quantile(0.975)}).round(4))

test texts identical to a training text: 0.539

zero-shot


,value,95% low,95% high
pr_auc,0.0584,0.0555,6.100000e-02
recall,0.0002,0.0000,5.000000e-04
precision,0.0004,0.0000,1.200000e-03
f1,0.0002,0.0000,7.000000e-04
false_alarm_rate,0.0410,0.0354,4.600000e-02
alerts_per_real_attack,2604.1310,855.3772,2.908205e+12



fine-tuned


,value,95% low,95% high
pr_auc,0.9868,0.9766,0.9959
recall,0.9984,0.9973,0.9993
precision,0.8289,0.8025,0.8586
f1,0.9058,0.8898,0.9233
false_alarm_rate,0.0184,0.0146,0.0218
alerts_per_real_attack,1.2064,1.1646,1.2461


In [ ]:
## how much of the test set was seen in training, and scores on the unseen part only
seen = test_set.text.isin(set(train_set.text))
print("share identical to a training text, by class:")
display(pd.crosstab(test_set.attack.map({True: "attack", False: "normal"}), seen.map({True: "seen", False: "unseen"}),
                    normalize="index").round(3))

unseen = test_set[~seen]
print(f"\nunseen test connections: {len(unseen):,} ({int(unseen.attack.sum()):,} attacks)")
display(pd.Series(detection(tuned_pred, unseen)).round(4))
display(pd.Series(score_predictions(tuned_pred.loc[unseen.index], unseen)).round(4))

share identical to a training text, by class:


text,seen,unseen
attack,,
attack,0.577,0.423
normal,0.518,0.482



unseen test connections: 3,630 (1,218 attacks)


,0
pr_auc,0.9820
recall,0.9974
precision,0.5955
f1,0.7457
false_alarm_rate,0.0344
alerts_per_real_attack,1.6794


,0
parsed_answers,1.0000
answers_with_invalid_labels,0.0000
attack_pr_auc,0.9820
attack_roc_auc,0.9991
attack_recall,0.9974
attack_precision,0.5955
false_alarm_rate,0.0344
tactics_exact_match,0.9450
tactics_first_correct,0.9548
tactics_micro_f1,0.9559


## 11. Next
- Test file arrives: put it at `TEST_FILE`, restart the runtime and run all cells. Training is skipped (the saved adapter
  is loaded), and zero-shot and fine-tuned scores are written to `OUT_DIR`.
- Quick end-to-end check first: `PILOT = True` (writes to its own `__pilot` folder).
- Leave-one-dataset-out instead: `SPLIT = "leave_one_dataset_out"` and `HOLD_OUT`, rerun from section 2.
- Model 4: serve the saved adapter behind an OpenAI-compatible server and point the backend at it (`LLM_PROVIDER=local`).